In [ ]:
%pip install openai
%pip install pydantic-settings
%pip install tiktoken

In [ ]:
from pydantic_settings import BaseSettings


class Env(BaseSettings):
    AZURE_OPENAI_ENDPOINT: str
    AZURE_OPENAI_API_KEY: str
    AZURE_OPENAI_API_VERSION: str = "2023-12-01-preview"
    GPT_35_MODEL_NAME: str = "gpt-35-turbo"
    GPT_4_MODEL_NAME: str = "gpt-4"
    TEXT_EMBEDDING_MODEL_NAME: str = "text-embedding-ada-002"

    class Config:
        env_file = 'streaming.env'


env = Env()

## Streaming

In [ ]:
from openai import AzureOpenAI

openai = AzureOpenAI(azure_endpoint=env.AZURE_OPENAI_ENDPOINT, api_key=env.AZURE_OPENAI_API_KEY,
                     api_version=env.AZURE_OPENAI_API_VERSION)

In [ ]:
prompts = [
    {"role": "user", "content": "請給我 50 字的自我介紹"}
]

response = openai.chat.completions.create(
    model=env.GPT_35_MODEL_NAME,
    messages=prompts,
    temperature=0,
    stream=True,
)

collected_chunks = []
collected_messages = []
# iterate through the stream of events
for chunk in response:
    if chunk.object != "chat.completion.chunk":
        continue
    collected_chunks.append(chunk)  # save the event response
    chunk_message = chunk.choices[0].delta  # extract the message
    
    if chunk_message.content:
        prompts.append(chunk_message)  # add the message to the prompt
        collected_messages.append(chunk_message)  # save the message
        print(f"Received message: {chunk_message.content}")

full_reply_content = ''.join(list(map(lambda m: m.content, collected_messages)))
print(f"Full conversation received: {full_reply_content}")
